# -- PointNet - Introduction to Point Cloud deep learning --

In [ ]:
# If working on google colab uncomment the following line to clone the repository
# !git clone https://github.com/JohnRomanelis/Lab08.git 
# !cp -r Lab08/utils .

# Exploring the notebook (Google Colab) environment

In [ ]:
import torch
torch.__version__

## Installing Libraries in Colab


In [ ]:
!pip install -q torch-geometric

In [ ]:
import torch_geometric as tgm
tgm.__version__

# Data


In [ ]:
from torch_geometric.datasets import ModelNet
import time

t1 = time.time()
modelnet = ModelNet(root='data/ModelNet10', # Where the data will be stored
                    name='10',  # Which version of Model we want to use - ModelNet10 or ModelNet40
                    train=True  # Deep learning datasets are split in different subsets, 'training', 'evaluation', 'testing'
                    )

print(f"Time to download and process data: {(time.time() - t1)/60} min") # Should take around 8 minutes

In [ ]:
# total number of elements in a dataset


In [ ]:
# accessing a single element from the dataset


In [ ]:
modelnet[0].pos[:3]

In [ ]:
modelnet[0].face[:, :3]

In [ ]:
# Visualizing a mesh
from utils.vis import visualize_tgm_mesh
visualize_tgm_mesh(modelnet[0])

In [ ]:
import torch_geometric.transforms as T

# Sample 1024 points on the surface of each mesh
transform = T.SamplePoints(num=1024)

modelnet_pc = ModelNet(
    root='data/ModelNet10',
    name='10',
    train=True,
    transform=transform
)

In [ ]:
modelnet_pc[0]

In [ ]:
from utils.vis import visualize_point_cloud
visualize_point_cloud(modelnet_pc[0].pos)

## Creating a custom Dataset


In [ ]:
point_clouds = []
labels = []
for item in modelnet_pc:
  pc = item.pos
  label = item.y
  point_clouds.append(pc)
  labels.append(label)


In [ ]:
len(point_clouds), len(labels)

In [ ]:
import numpy as np

point_clouds = np.stack(point_clouds)
labels = np.stack(labels)

In [ ]:
point_clouds.shape, labels.shape

In [ ]:
# Save data on the disk
np.save('data/train_point_clouds.npy', point_clouds)
np.save('data/train_labels.npy', labels)

### Custom Dataset Class

In [ ]:
from torch.utils.data import Dataset

## Create a custom dataset class for our point clouds

In [ ]:
custom_modelnet = CustomModelNet10(path='data/')

In [ ]:
custom_modelnet[0]

In [ ]:
pc, label = custom_modelnet[0]

### Transforms


In [ ]:
class CustomModelNet10(Dataset):

  def __init__(self, path, transforms = []):
    super().__init__()

    self.point_clouds = np.load(path + 'train_point_clouds.npy')
    self.labels = np.load(path + 'train_labels.npy')

    # Add the ability to apply transforms on the data
    

  def __len__(self):
    return len(self.point_clouds)

  def __getitem__(self, index):
    pc = self.point_clouds[index]
    label = self.labels[index]

    # Apply transforms on the data
    

    return pc, label

In [ ]:
class NumpyToTorch:
  
  # TODO:
  

In [ ]:
# Pass teh transform to the dataset

In [ ]:
pc, label = custom_modelnet[0]
type(pc), type(label)

## Let's summarize...


In [ ]:
## Load the dataset using ModelNet from torch geometric -- same transform as before
modelnet_pc_test = ModelNet(
    root='data/ModelNet10',
    name='10',
    train=False,
    transform=transform
)

## Save data on the disk
point_clouds = []
labels = []
for item in modelnet_pc:
  pc = item.pos
  label = item.y
  point_clouds.append(pc)
  labels.append(label)

point_clouds = np.stack(point_clouds)
labels = np.stack(labels)

np.save('data/test_point_clouds.npy', point_clouds)
np.save('data/test_labels.npy', labels)


In [ ]:
class CustomModelNet10(Dataset):

  def __init__(self, path, split='train', transforms = []):
    super().__init__()

    assert split in ['train', 'test'], 'Invalid split'

    self.split = split

    self.point_clouds = np.load(path + self.split + '_point_clouds.npy')
    self.labels = np.load(path + self.split + '_labels.npy')

    # Handle single transform
    if not isinstance(transforms, (list, tuple)):
      transforms = [transforms]

    self.transforms = transforms

  def __len__(self):
    return len(self.point_clouds)

  def __getitem__(self, index):
    pc = self.point_clouds[index]
    label = self.labels[index]

    # applying transforms on the data
    for t in self.transforms:
      pc, label = t(pc, label)

    return pc, label

In [ ]:
train_dataset = CustomModelNet10(path='data/', split='train', transforms=NumpyToTorch())
test_dataset = CustomModelNet10(path='data/', split='test', transforms=NumpyToTorch())

## Dataloaders


In [ ]:
from torch.utils.data import DataLoader

# TODO: Create dataloaders for the train and test datasets


In [ ]:
# TODO: Get a batch of data from the dataloader and check their shapes

# PointNet


### Some torch documentation notes:


In [ ]:
import torch.nn as nn

In [ ]:
?? nn.Linear

In [ ]:
# Define a linear layer that takes 3D points as input and outputs 16-dimensional features

In [ ]:
# Let's create a random point cloud
pc = torch.randn(1024, 3)
pc.shape

In [ ]:
# Let's pass the points through this linear layer


In [ ]:
# Adding a non-linear function
# activation = ...
print(pc_proj[0])
# pc_proj = ...
print(pc_proj[0])

In [ ]:
## Creating an MLP: Linear + ReLU + Linear (3->16->32)
linear2 = 

pc_proj = 
pc_proj = 
pc_proj = 

pc_proj.shape

In [ ]:
mlp = nn.Sequential(
    ...
)

pc_proj = mlp(pc)
pc_proj.shape

## BatchNorm


In [ ]:
mlp = nn.Sequential(
    nn.Linear(3, 16),
    # TODO: normalization layer
    nn.ReLU(),
    nn.Linear(16, 32)
)

pc_proj = mlp(pc)
pc_proj.shape

### What about the batch dimension


In [ ]:
# Let's create a random batch of 2 point clouds
batch = torch.randn(2, 1024, 3)

In [ ]:
batch_proj = linear(batch)
batch_proj.shape

In [ ]:
norm = nn.BatchNorm1d(16)
batch_proj = norm(batch_proj)
batch_proj.shape

### Why this didnt work:


In [ ]:
batch_proj = ...
batch_proj.shape

In [ ]:
batch_proj = norm(batch_proj)
batch_proj.shape

In [ ]:
batch = batch.permute(0, 2, 1)
batch.shape

In [ ]:
mlp = nn.Sequential(
    # Replace Linear Layer ...
    nn.BatchNorm1d(16),
    nn.ReLU(),
    # Replace Linear Layer ...
)

In [ ]:
batch_proj = mlp(batch)
batch_proj.shape

In [ ]:
class LinearLayer(nn.Module): # Always inherit nn.Module
  def __init__(self, in_features, out_features):
    super().__init__() # Always call the parent __init__

    self.layers = nn.Sequential(
        nn.Conv1d(in_features, out_features, kernel_size=1),
        nn.BatchNorm1d(out_features),
        nn.ReLU()
    )

  def forward(self, x): # This is what code is excecuted when activating the network.
    return self.layers(x)


In [ ]:
linear_layer = LinearLayer(3, 16)
batch_proj = linear_layer(batch)
batch_proj.shape

## Pooling Layers
### Common Aggregation Strategies


In [ ]:
batch_proj.shape

In [ ]:
## Let's create a single vector for each point cloud in the batch
# batch_proj is of shape (B, F, N) --> we apply the reduction over the last dimension
batch_vector = ... # Apply max pooling 
batch_vector.shape

In [ ]:
# Create cls mlp 16-> 32 -> 10
cls_mlp = nn.Sequential(
    
    # 10 is the number of classes
)

out = cls_mlp(batch_vector)
out.shape

### Simple PointNet


In [ ]:
class SimplePointNet(nn.Module):
  def __init__(self, num_classes):
    super().__init__()

    self.mlp1 = nn.Sequential(
        LinearLayer(3, 64),
        LinearLayer(64, 64)
    )

    self.mlp2 = nn.Sequential(
        LinearLayer(64, 64),
        LinearLayer(64, 128),
        LinearLayer(128, 1024)
    )


    self.cls_head = nn.Sequential(
        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.3), # --> Food for thought...
        nn.Linear(256, num_classes)
    )


  def forward(self, x):

    x = self.mlp1(x)

    x = self.mlp2(x)

    # Aggregation
    x_vector = x.max(dim=-1)[0]

    out = self.cls_head(x_vector)

    return out

In [ ]:
model = SimplePointNet(num_classes=10)

In [ ]:
prediction = model(batch)
prediction.shape

# Training the network


## The Training Objective: The Loss Function
### Theoretical Background
### Understanding the Output and the Loss Function
### L1 or L2 loss
### Cross-Entropy loss


In [ ]:
# Defining the loss function - aka criterion
criterion = ...

In [ ]:
# Creating some fake labels for our dummy point cloud
dummy_labels = torch.tensor([0, 2])

In [ ]:
loss = ...
loss

## Updating the Weights:
### 1. Gradient Computation


In [ ]:
# Compute the gradients


In [ ]:
# we can manually access the gradient values of the network weights
model.mlp1[0].layers[0].weight.grad.shape

### 2. Updating the weights


In [ ]:
import torch.optim as optim

In [ ]:
# Create an optimizer

In [ ]:
# Update the weights 


#### 🚨 Important: Resetting Gradients


In [ ]:
optimizer.zero_grad() # --> Deletes the gradient!
model.mlp1[0].layers[0].weight.grad == None

## Training Loop

In [ ]:
from tqdm import tqdm # progress bar

In [ ]:
epochs = 1
lr = 0.01

model = SimplePointNet(num_classes=10)
optimizer = optim.SGD(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
  # ============================
  #       TRAINING PHASE
  # ============================

  # ...  
  running_loss = 0.0  # To compute the average loss per epoch
  correct_train = 0   # Correctly classified samples
  total_train = 0     # Total samples



  for batch in tqdm(train_dataloader):
    



  epoch_train_loss = running_loss / total_train
  epoch_train_acc = correct_train / total_train * 100
  print(f"-> [Summary] Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")

  # ============================
  #       VALIDATION PHASE
  # ============================

  # ...
  running_val_loss = 0.0
  correct_val = 0
  total_val = 0

  # ...
    # for batch in tqdm(test_dataloader):
      # ...



    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = (correct_val / total_val) * 100

    print(f"   ✨ [Validation Results] Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")
    print("-" * 60)

## Running on the GPU!

In [ ]:
epochs = 1
lr = 0.01

device = ...

print("Using device: ", device)

model = SimplePointNet(num_classes=10) # ...
optimizer = optim.SGD(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
  # ============================
  #       TRAINING PHASE
  # ============================
  model.train() # <-- Important for layers like BatchNorm

  running_loss = 0.0  # To compute the average loss per epoch
  correct_train = 0   # Correctly classified samples
  total_train = 0     # Total samples



  for batch in tqdm(train_dataloader):
    x, y = batch

    x = x.permute(0, 2, 1) # x.shape (B, N, F) --> (B, F, N)
    y = y.squeeze() # y.shape (B, 1) --> (B,)
    optimizer.zero_grad()
    prediction = model(x)
    loss = criterion(prediction, y)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * x.shape[0] # loss is averaged across the batch

    _, predicted_classes = torch.max(prediction, 1)
    total_train += x.shape[0]
    correct_train += (predicted_classes == y).sum().item()

  epoch_train_loss = running_loss / total_train
  epoch_train_acc = correct_train / total_train * 100
  print(f"-> [Summary] Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")

  # ============================
  #       VALIDATION PHASE
  # ============================
  model.eval()

  running_val_loss = 0.0
  correct_val = 0
  total_val = 0

  with torch.no_grad():
    for batch in tqdm(test_dataloader):
      x, y = batch

      x = x.permute(0, 2, 1)
      y = y.squeeze()

      prediction = model(x)
      val_loss = criterion(prediction, y)

      # --- Metrics Tracking ---
      running_val_loss += val_loss.item() * x.size(0)
      _, predicted_classes = torch.max(prediction, dim=1)
      correct_val += (predicted_classes == y).sum().item()
      total_val += y.size(0)

    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = (correct_val / total_val) * 100

    print(f"   ✨ [Validation Results] Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")
    print("-" * 60)

# Data Proprocessing


In [ ]:
pc1 = custom_modelnet[0][0]
pc1.min(), pc1.max()

In [ ]:
pc2 = custom_modelnet[-1][0]
pc2.min(), pc2.max()

In [ ]:
# Create a unit sphere normalization transform

class UnitSphereNormalize:

  def __call__(self, x, y):
    # x.shape N, 3


    return x, y

In [ ]:
train_dataset = CustomModelNet10(path='data/', split='train', transforms=[....])
test_dataset = CustomModelNet10(path='data/', split='test', transforms=[....])

In [ ]:
pc1 = train_dataset[0][0]
pc1.min(), pc1.max()

In [ ]:
pc2 = test_dataset[-1][0]
pc2.min(), pc2.max()

### Train with normalized data

In [ ]:
epochs = 1
lr = 0.01

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Using device: ", device)

model = SimplePointNet(num_classes=10).to(device)
optimizer = optim.SGD(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
  # ============================
  #       TRAINING PHASE
  # ============================
  model.train() # <-- Important for layers like BatchNorm

  running_loss = 0.0  # To compute the average loss per epoch
  correct_train = 0   # Correctly classified samples
  total_train = 0     # Total samples



  for batch in tqdm(train_dataloader):
    x, y = batch
    x = x.to(device)
    y = y.to(device)

    x = x.permute(0, 2, 1) # x.shape (B, N, F) --> (B, F, N)
    y = y.squeeze() # y.shape (B, 1) --> (B,)
    optimizer.zero_grad()
    prediction = model(x)
    loss = criterion(prediction, y)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * x.shape[0] # loss is averaged across the batch

    _, predicted_classes = torch.max(prediction, 1)
    total_train += x.shape[0]
    correct_train += (predicted_classes == y).sum().item()

  epoch_train_loss = running_loss / total_train
  epoch_train_acc = correct_train / total_train * 100
  print(f"-> [Summary] Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")

  # ============================
  #       VALIDATION PHASE
  # ============================
  model.eval()

  running_val_loss = 0.0
  correct_val = 0
  total_val = 0

  with torch.no_grad():
    for batch in tqdm(test_dataloader):
      x, y = batch
      x = x.to(device)
      y = y.to(device)

      x = x.permute(0, 2, 1)
      y = y.squeeze()

      prediction = model(x)
      val_loss = criterion(prediction, y)

      # --- Metrics Tracking ---
      running_val_loss += val_loss.item() * x.size(0)
      _, predicted_classes = torch.max(prediction, dim=1)
      correct_val += (predicted_classes == y).sum().item()
      total_val += y.size(0)

    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = (correct_val / total_val) * 100

    print(f"   ✨ [Validation Results] Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")
    print("-" * 60)